<a href="https://colab.research.google.com/github/max7082/colab-practice/blob/main/%EC%9C%A4%ED%98%84%EC%9B%85_%EA%B5%AC%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import torch
from torch import nn
from torchvision.datasets import MNIST

device = "cuda" if torch.cuda.is_available() else "cpu"
data = MNIST("./data", train=True, download=True)

x = data.data[:2000].float().reshape(2000, 784).to(device) / 255
label = data.targets[:2000].to(device)
y = nn.functional.one_hot(label, num_classes=10).float()
val_x = data.data[2000:3000].float().reshape(1000, 784).to(device) / 255
val_y = data.targets[2000:3000].to(device)

#네 가지 방법 비교
for method in ["ERM", "입력만 혼합", "정답만 혼합", "mixup"]:
    torch.manual_seed(44)
    model = nn.Sequential(
        nn.Linear(784, 64),
        nn.ReLU(),
        nn.Linear(64, 10)
    ).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

    for _ in range(500):
        lam = torch.distributions.Beta(0.2, 0.2).sample().item()
        j = torch.randperm(len(x), device=device)
        mixed_x = lam * x + (1 - lam) * x[j]
        mixed_y = lam * y + (1 - lam) * y[j]
        if method == "ERM":
            input_x = x
            target_y = y
        elif method == "입력만 혼합":
            input_x = mixed_x
            target_y = y if lam >= 0.5 else y[j]
        elif method == "정답만 혼합":
            input_x = x
            target_y = mixed_y
        else:
            input_x = mixed_x
            target_y = mixed_y
        loss = nn.functional.cross_entropy(model(input_x), target_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        prediction = model(val_x).argmax(1)
        error = (prediction != val_y).float().mean().item()

    print(f"{method}: 오류율 {error:.1%}")

ERM: 오류율 9.4%
입력만 혼합: 오류율 10.0%
정답만 혼합: 오류율 11.4%
mixup: 오류율 9.5%
